# 03 — IGV Design

**Purpose:** Perform the complete Inlet Guide Vane aerodynamic design. Auto-size rotor blade count B and mid-span chord; select IGV count V with gcd(B,V)=1 to suppress locked Tyler–Sofrin interaction modes; apply Zweifel loading and Carter deviation to set IGV camber and stagger; verify Lieblein DF for the IGV row; establish the rotor inlet velocity triangle.

**Inputs:** Design point fixed in notebook 01.

**Outputs:**
- Annulus dimensions, blade counts B and V
- IGV camber, stagger, solidity, axial chord
- Rotor inlet β₁ at hub / mean / tip
- Pre-swirl sensitivity table
- Axial station positions

**References:** Dixon & Hall (2014); Cumpsty (2004) Ch. 5; Zweifel (1945); Carter (1950) ARC R&M 2804; Tyler & Sofrin (1962).

In [ ]:
import sys, pathlib

# Locate repo root (directory that contains src/) regardless of
# where the notebook file sits (repo root or notebooks/ subfolder)
_here = pathlib.Path.cwd()
_root = next(
    (p for p in [_here, *_here.parents] if (p / "src").is_dir()),
    _here,
)
sys.path.insert(0, str(_root))
pathlib.Path(_root / "figures").mkdir(exist_ok=True)


import numpy as np
import matplotlib.pyplot as plt
import math

from src.igv import igv_geometry, meanline_with_igv, print_igv_summary, print_rotor_summary

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})
print('Imports OK')

In [ ]:
# ── Load design point from notebook 01 ─────────────────────
import json as _json, pathlib as _pl

_dp_path = _pl.Path(_root / "design_point.json")
if not _dp_path.exists():
    raise FileNotFoundError(
        "design_point.json not found — run notebook 01 first.")

dp = _json.loads(_dp_path.read_text())

# Unpack for convenience
D_TIP = dp["D_tip"]
N_RPM = dp["N_RPM"]
PR    = dp["PR"]
NU    = dp["nu"]
PHI   = dp["phi"]
ETA   = dp["eta_is"]
print(f"Design point: D_tip={D_TIP*1000:.0f} mm  N={N_RPM} RPM  PR={PR}  nu={NU}  phi={PHI}  eta={ETA}")


## 1. Design point IGV geometry

In [ ]:
igv = igv_geometry(
    D_tip      = D_TIP,
    nu         = NU,
    N_RPM      = N_RPM,
    phi        = PHI,
    alpha1_deg = 0.0,   # axial inlet — zero pre-swirl at design point
)
print_igv_summary(igv)


## 2. Rotor meanline at design point

In [ ]:
rotor = meanline_with_igv(igv, PR=PR, eta_is=ETA)
print_rotor_summary(rotor)

## 3. Tyler–Sofrin interaction mode check

In [ ]:
B = igv['B_blades']
V = igv['V_blades']
print(f'Rotor blades B = {B},  IGV blades V = {V},  gcd(B,V) = {igv["gcd_BV"]}')
print()
print('Tyler–Sofrin modes  m = nB + kV  (lowest |m| first):')
print(f'  {"n":>3} {"k":>4} {"m":>6}')
for n, k, m in igv['ts_modes']:
    flag = '  ← LOCKED (m=0)' if m == 0 else ''
    print(f'  {n:>3} {k:>4} {m:>6}{flag}')
print(f'\nLowest non-zero |m| = {min(abs(x[2]) for x in igv["ts_modes"] if x[2]!=0)}')

## 4. Pre-swirl sensitivity

Sweep IGV exit angle α₁ from –15° to +15° to understand how pre-swirl affects rotor loading and inlet relative angle.

In [ ]:
alpha_range = np.linspace(-15, 15, 13)
rows = []

print(f'  {"α₁ [°]":>7} {"β₁ [°]":>8} {"W₁ [m/s]":>10} {"ψ":>8} {"DH":>8} {"DF":>8} {"P [kW]":>8}')
print('  ' + '─'*62)

for a1 in alpha_range:
    g  = igv_geometry(D_tip=D_TIP, nu=NU, N_RPM=N_RPM, phi=PHI, alpha1_deg=a1)
    rr = meanline_with_igv(g, PR=PR, eta_is=ETA)
    ok = '✓' if rr['DH_ok'] and rr['DF_ok'] else '⚠'
    print(f'  {a1:>7.1f} {rr["beta1_deg"]:>8.2f} {rr["W1_m_s"]:>10.2f}'
          f' {rr["psi"]:>8.4f} {rr["De_Haller"]:>8.4f}'
          f' {rr["DF_rotor"]:>8.4f} {rr["P_shaft_kW"]:>8.1f}  {ok}')
    rows.append((a1, rr['beta1_deg'], rr['W1_m_s'], rr['psi'],
                 rr['De_Haller'], rr['DF_rotor'], rr['P_shaft_kW']))

In [ ]:
rows = np.array(rows)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('Pre-swirl Sensitivity — α₁ sweep', fontweight='bold')

axes[0].plot(rows[:,0], rows[:,1], '#185FA5', lw=2)
axes[0].set(xlabel='IGV exit angle α₁ [°]', ylabel='Rotor inlet β₁ [°]')
axes[0].axvline(0, color='gray', ls=':')

axes[1].plot(rows[:,0], rows[:,4], '#1D9E75', lw=2, label='De Haller')
axes[1].axhline(0.72, color='red', ls='--', label='DH limit')
axes[1].plot(rows[:,0], rows[:,5], '#D85A30', lw=2, ls='--', label='DF')
axes[1].axhline(0.45, color='orange', ls=':', label='DF limit')
axes[1].set(xlabel='IGV exit angle α₁ [°]', ylabel='Coefficient')
axes[1].legend(fontsize=9)

axes[2].plot(rows[:,0], rows[:,6], '#533AB7', lw=2)
axes[2].set(xlabel='IGV exit angle α₁ [°]', ylabel='Shaft power [kW]')

for ax in axes:
    ax.grid(True, alpha=0.35)
    ax.axvline(0, color='gray', ls=':')
plt.tight_layout()
plt.savefig(str(_root / 'figures') + '/03_preswirl_sensitivity.png', dpi=130, bbox_inches='tight')
plt.show()

## 5. Spanwise velocity triangles at rotor inlet

Show how β₁ varies from hub to tip as computed by the IGV free-vortex exit.

In [ ]:
stations = igv['stations']
labels   = ['hub', 'mean', 'tip']
colors   = ['#D85A30', '#185FA5', '#1D9E75']

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.suptitle('Rotor Inlet Velocity Triangles — hub / mean / tip', fontweight='bold')

for ax, lbl, col in zip(axes, labels, colors):
    st = stations[lbl]
    Ca = st['Ca_m_s']
    U  = st['U_m_s']
    Ct = st['C_theta1']
    # Vectors: Ca (axial), Ct (tangential), U (blade speed)
    # Absolute: C1 = (Ca, Ct)
    # Relative: W1 = (Ca, Ct-U)
    ax.annotate('', xy=(Ca, 0), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='k', lw=1.5))
    ax.annotate('', xy=(Ca, Ct), xytext=(Ca, 0),
                arrowprops=dict(arrowstyle='->', color='#185FA5', lw=1.5))
    ax.annotate('', xy=(Ca, Ct-U), xytext=(Ca, 0),
                arrowprops=dict(arrowstyle='->', color='#D85A30', lw=1.5))
    ax.annotate('', xy=(0, -U), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='#854F0B', lw=1.5))
    ax.set(xlabel='Axial [m/s]', ylabel='Tangential [m/s]',
           title=f'{lbl.capitalize()}  r={st["r_mm"]:.0f} mm\n'
                 f'β₁={st["beta1_deg"]:.1f}°  α₁={st["alpha1_deg"]:.1f}°')
    ax.axhline(0, color='gray', lw=0.7)
    ax.axvline(0, color='gray', lw=0.7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(_root / 'figures') + '/03_inlet_velocity_triangles.png', dpi=130, bbox_inches='tight')
plt.show()

## 6. Axial station map

In [ ]:
print('Axial station map  (x = 0 at rotor LE, upstream = negative)')
print('─' * 55)
print(f'  IGV TE → rotor LE gap : {igv["axial_gap_mm"]:.1f} mm')
print(f'  IGV axial chord       : {igv["igv_axial_len_mm"]:.1f} mm')
print(f'  IGV LE position       : x = {-igv["igv_LE_to_rotor_LE_mm"]:.1f} mm')
print(f'  IGV TE position       : x = {-igv["axial_gap_mm"]:.1f} mm')
print(f'  Rotor LE              : x = 0 mm  ◀ reference')

---
**Proceed to** `04_inlet_bellmouth.ipynb`.